# Full pipeline example: scraping -> evaluation metrics

Runs every stage of this project's pipeline end to end for one dataset,
using the real modules built throughout the project (not reimplemented
here): `scraper.py` -> `chunk_documents.py` -> `generate_queries.py` ->
`extract_topics.py` -> `topic_to_queries.py` -> `chunks_to_docs.py` ->
sampling -> `pt_retrieval.py` / `pt_dense_retrieval.py` -> `pool_results.py`
-> `judge_pool.py` -> `evaluate_run.py`.

**Every stage is idempotent**: it loads the existing output file if one is
already on disk, and only computes it otherwise. Several stages are slow
LLM/embedding jobs (per-chunk query generation, topic naming, LLM-as-judge
labelling can each take from minutes to hours on a full dataset), so this
notebook is safe to re-run -- it will skip straight past anything already
done and only do the work that's actually missing.

**Dataset for this run: fln.dk / Flygtningenaevnet (`fln`), on the 30-topic
sample** used throughout this project, since that's the topic set that
already has a fully judged qrels pool.

**Kernel:** must run under a Python distribution where PyTerrier's embedded
JVM actually starts (Anaconda's, on this machine) -- retrieval and
evaluation need it; the earlier stages don't.

## 0. Setup

In [1]:
import os, sys, random
import pandas as pd

PROJECT_DIR = os.path.abspath(".")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

pd.set_option("display.max_colwidth", 80)

DATASET = "fln"
BASE_URL = "https://fln.dk/praksis/"
SCRAPE_CSV = "fln_praksis_2026.csv"
CHUNKS_CSV = "fln_praksis_2026_chunks.csv"
QUERIES_CSV = "fln_praksis_2026_queries.csv"
TOPIC_NAMES_CSV = "fln_praksis_2026_queries_topic_names.csv"
TOPICS_QID_QUERY_CSV = "fln_praksis_2026_queries_topic_names_qid_query.csv"
CHUNKS_DOCS_CSV = "fln_praksis_2026_chunks_docs.csv"
SAMPLE_SIZE = 30
SAMPLE_SEED = 42
SAMPLE_TOPICS_CSV = f"fln_praksis_2026_queries_topic_names_qid_query_sample{SAMPLE_SIZE}.csv"
RETRIEVERS = ["bm25", "bm25_monot5", "splade", "e5", "qwen3"]
POOL_CSV = f"fln_praksis_2026_queries_topic_names_qid_query_sample{SAMPLE_SIZE}_pool.csv"
LABELS_CSV = f"fln_praksis_2026_queries_topic_names_qid_query_sample{SAMPLE_SIZE}_pool_labels.csv"

def step_done(path):
    return os.path.exists(path) and os.path.getsize(path) > 0

## 1. Scrape the source site

`scraper.py`'s `WebsiteScraper`, configured for fln.dk/praksis/ (`FLN_PRAKSIS`).

In [2]:
from scraper import FLN_PRAKSIS, WebsiteScraper

if not step_done(SCRAPE_CSV):
    print(f"Scraping {BASE_URL} ...")
    WebsiteScraper(FLN_PRAKSIS).run(SCRAPE_CSV)
else:
    print(f"Already scraped -> {SCRAPE_CSV} (skipping)")

df_scraped = pd.read_csv(SCRAPE_CSV)
print(f"{len(df_scraped)} scraped cases")
df_scraped.head(3)

Already scraped -> fln_praksis_2026.csv (skipping)
284 scraped cases


,item_id,title,url,published_date,categories,description
0,afgh202618,afgh202618,https://fln.dk/praksis/2026/september/afgh202618/,2026-09-08,Afghanistan (I)|2026|8|Sur Place|Genoptagede sager,Nævnet meddelte i august 2026 opholdstilladelse (K-status) til en mandlig st...
1,soma202611,soma202611,https://fln.dk/praksis/2026/september/soma202611/,2026-09-08,Somalia (I)|2026|8|Generelle forhold|Helbredsmæssige forhold|Køns- og æresre...,Nævnet stadfæstede i august 2026 Udlændingestyrelsens afgørelse en mandlig s...
2,syri202620,syri202620,https://fln.dk/praksis/2026/september/syri202620/,2026-09-08,Syrien (I)|2026|8|Etniske forhold|Køns- og æresrelateret forfølgelse - Seksu...,Nævnet meddelte i august 2026 opholdstilladelse (B-status) til en kvindelig ...


## 2. Chunk the scraped documents

`chunk_documents.py`'s `DocumentChunker`: splits each case at natural
paragraph/sentence boundaries into ~400-char chunks.

In [3]:
from chunk_documents import ChunkingConfig, DocumentChunker

if not step_done(CHUNKS_CSV):
    config = ChunkingConfig(input_csv=SCRAPE_CSV, text_column="description", output_csv=CHUNKS_CSV)
    DocumentChunker(config).run()
else:
    print(f"Already chunked -> {CHUNKS_CSV} (skipping)")

df_chunks = pd.read_csv(CHUNKS_CSV)
print(f"{df_chunks['doc_id'].nunique()} documents -> {len(df_chunks)} chunks")
df_chunks.head(3)

Already chunked -> fln_praksis_2026_chunks.csv (skipping)
281 documents -> 5844 chunks


,chunk_id,doc_id,chunk_index,total_chunks,chunk_text,item_id,title,url,published_date,categories
0,afgh202618_chunk000,afgh202618,0,19,Nævnet meddelte i august 2026 opholdstilladelse (K-status) til en mandlig st...,afgh202618,afgh202618,https://fln.dk/praksis/2026/september/afgh202618/,2026-09-08,Afghanistan (I)|2026|8|Sur Place|Genoptagede sager
1,afgh202618_chunk001,afgh202618,1,19,"Herefter vendte de tilbage til Kabul, hvor de boede indtil udrejsen i [slut ...",afgh202618,afgh202618,https://fln.dk/praksis/2026/september/afgh202618/,2026-09-08,Afghanistan (I)|2026|8|Sur Place|Genoptagede sager
2,afgh202618_chunk002,afgh202618,2,19,"Ansøgeren har til støtte herfor oplyst, at han ikke har været i Afghanistan,...",afgh202618,afgh202618,https://fln.dk/praksis/2026/september/afgh202618/,2026-09-08,Afghanistan (I)|2026|8|Sur Place|Genoptagede sager


## 3. Generate one query per chunk (few-shot, local LLM)

`generate_queries.py`'s `QueryGenerator`, via a local Ollama model
(`gemma3:4b` default) and the few-shot prompt in `fewshot_qgen.txt` /
`fewshot_examples.txt`.

**Slow on a full dataset** (this took ~85 minutes for all 5,844 fln
chunks) -- skipped here since it's already done.

In [4]:
from generate_queries import QueryGenConfig, QueryGenerator

if not step_done(QUERIES_CSV):
    print("Generating one query per chunk via Ollama (this can take a while)...")
    config = QueryGenConfig(input_csv=CHUNKS_CSV, output_csv=QUERIES_CSV)
    QueryGenerator(config).run()
else:
    print(f"Already generated -> {QUERIES_CSV} (skipping)")

df_queries = pd.read_csv(QUERIES_CSV)
print(f"{len(df_queries)} generated queries")
df_queries[["chunk_id", "generated_query"]].head(3)

Already generated -> fln_praksis_2026_queries.csv (skipping)
5844 generated queries


,chunk_id,generated_query
0,afgh202618_chunk000,provide a definition of credibility assessment
1,afgh202618_chunk001,what does the applicant fear from the Taliban?
2,afgh202618_chunk002,what is the applicant’s background and lifestyle?


## 4. Cluster queries into topics (BERTopic) and name each with an LLM

`extract_topics.py`'s `QueryTopicExtractor`: sentence-transformers embedding
-> UMAP -> HDBSCAN -> c-TF-IDF keywords -> Ollama-generated topic name.
Produces both the per-query assignment and the reduced topic-level table.

In [5]:
from extract_topics import TopicExtractionConfig, QueryTopicExtractor

per_query_topics_csv = QUERIES_CSV.replace(".csv", "_topics.csv")

if not (step_done(TOPIC_NAMES_CSV) and step_done(per_query_topics_csv)):
    print("Embedding, clustering, and naming topics (this can take a minute or two)...")
    config = TopicExtractionConfig(input_csv=QUERIES_CSV)
    QueryTopicExtractor(config).run()
else:
    print(f"Already extracted -> {TOPIC_NAMES_CSV} (skipping)")

df_topic_names = pd.read_csv(TOPIC_NAMES_CSV)
print(f"{len(df_topic_names)} topics found")
df_topic_names.sort_values("num_queries", ascending=False).head(5)

Already extracted -> fln_praksis_2026_queries_topic_names.csv (skipping)
137 topics found


,topic_id,topic_name,topic_keywords,num_queries
0,-1,Outliers,"applicant, to, the, for, in, and, of, asylum, what, an",1389
1,0,ukraine military recruitment cluster,"military, recruitment, service, ukraine, chechen, chechnya, russia, war, con...",365
2,1,Applicant Family Legal Disputes,"describe, police, threats, applicant, torture, spouse, relationship, between...",295
3,2,Credibility Assessment Definitions,"credibility, assessment, definition, provide, of, context, narratives, narra...",284
4,3,flygtningenævnet asylum decisions,"flygtningenævnet, asylum, applications, decision, regarding, role, applicati...",84


## 5. Reduce topics to a qid/query table

`topic_to_queries.py`'s `TopicQuerySubsetter`: keeps just `topic_id` ->
`qid` and `topic_name` -> `query`, the shape retrieval topics need.

In [6]:
from topic_to_queries import TopicSubsetConfig, TopicQuerySubsetter

if not step_done(TOPICS_QID_QUERY_CSV):
    config = TopicSubsetConfig(input_csv=TOPIC_NAMES_CSV, output_csv=TOPICS_QID_QUERY_CSV)
    TopicQuerySubsetter(config).run()
else:
    print(f"Already reduced -> {TOPICS_QID_QUERY_CSV} (skipping)")

df_all_topics = pd.read_csv(TOPICS_QID_QUERY_CSV, dtype={"qid": str})
print(f"{len(df_all_topics)} topic/query rows")
df_all_topics.head(3)

Already reduced -> fln_praksis_2026_queries_topic_names_qid_query.csv (skipping)
137 topic/query rows


,qid,query
0,-1,Outliers
1,0,ukraine military recruitment cluster
2,1,Applicant Family Legal Disputes


## 6. Convert chunks to a docno/text index source

`chunks_to_docs.py`'s `ChunkDocsConverter`: `chunk_id`/`chunk_text` ->
`docno`/`text`, the column names PyTerrier indexing expects.

In [7]:
from chunks_to_docs import ChunkDocsConfig, ChunkDocsConverter

if not step_done(CHUNKS_DOCS_CSV):
    config = ChunkDocsConfig(input_csv=CHUNKS_CSV, output_csv=CHUNKS_DOCS_CSV)
    ChunkDocsConverter(config).run()
else:
    print(f"Already converted -> {CHUNKS_DOCS_CSV} (skipping)")

df_docs = pd.read_csv(CHUNKS_DOCS_CSV)
print(f"{len(df_docs)} indexable chunks")
df_docs.head(3)

Already converted -> fln_praksis_2026_chunks_docs.csv (skipping)
5844 indexable chunks


,docno,text
0,afgh202618_chunk000,Nævnet meddelte i august 2026 opholdstilladelse (K-status) til en mandlig st...
1,afgh202618_chunk001,"Herefter vendte de tilbage til Kabul, hvor de boede indtil udrejsen i [slut ..."
2,afgh202618_chunk002,"Ansøgeren har til støtte herfor oplyst, at han ikke har været i Afghanistan,..."


## 7. Sample 30 topics

A fixed, seeded random sample of the full topic set -- the query set this
whole example evaluates retrieval against.

In [8]:
if not step_done(SAMPLE_TOPICS_CSV):
    df_sample = df_all_topics.sample(n=SAMPLE_SIZE, random_state=SAMPLE_SEED).reset_index(drop=True)
    df_sample.to_csv(SAMPLE_TOPICS_CSV, index=False)
else:
    print(f"Already sampled -> {SAMPLE_TOPICS_CSV} (skipping)")

df_sample = pd.read_csv(SAMPLE_TOPICS_CSV, dtype={"qid": str})
print(f"{len(df_sample)} sampled topics")
df_sample.head(5)

Already sampled -> fln_praksis_2026_queries_topic_names_qid_query_sample30.csv (skipping)
30 sampled topics


,qid,query
0,104,Asylvurdering Ved Flygtningenævnet
1,103,health notifications and claimant status
2,11,Perceived Past Behavior & Plausibility
3,25,Eritrean Asylum Seekers' Journeys
4,122,Swedish Dublin Forordningen Respons


## 8. Retrieve: BM25, BM25+monoT5, SPLADE (sparse), E5, Qwen3 (dense)

`pt_retrieval.py`'s `PyTerrierRetrievalPipeline` (BM25/SPLADE, with the
Danish stemmer auto-applied for this collection, plus optional monoT5
rerank) and `pt_dense_retrieval.py`'s `DenseRetrievalPipeline` (E5/Qwen3
via FlexIndex). Each builds/reuses its own index and writes one CSV per
method; already-run methods are skipped.

In [9]:
from pt_retrieval import RetrievalConfig, PyTerrierRetrievalPipeline
from pt_dense_retrieval import DenseRetrievalConfig, DenseRetrievalPipeline

result_paths = {}

for retriever in ["bm25", "bm25_monot5", "splade"]:
    rerank = retriever.endswith("_monot5")
    base_retriever = "bm25" if rerank else retriever
    out_path = f"{os.path.splitext(SAMPLE_TOPICS_CSV)[0]}_{retriever}.csv"
    if not step_done(out_path):
        print(f"Running {retriever} ...")
        config = RetrievalConfig(
            docs_csv=CHUNKS_DOCS_CSV, topics_csv=SAMPLE_TOPICS_CSV,
            retriever=base_retriever, rerank_monot5=rerank, output_csv=out_path,
        )
        PyTerrierRetrievalPipeline(config).run()
    else:
        print(f"Already ran {retriever} -> {out_path} (skipping)")
    result_paths[retriever] = out_path

for retriever in ["e5", "qwen3"]:
    out_path = f"{os.path.splitext(SAMPLE_TOPICS_CSV)[0]}_{retriever}.csv"
    if not step_done(out_path):
        print(f"Running {retriever} ...")
        config = DenseRetrievalConfig(docs_csv=CHUNKS_DOCS_CSV, topics_csv=SAMPLE_TOPICS_CSV, retriever=retriever, output_csv=out_path)
        DenseRetrievalPipeline(config).run()
    else:
        print(f"Already ran {retriever} -> {out_path} (skipping)")
    result_paths[retriever] = out_path

result_paths

Running bm25 ...


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/Users/qbr926/Desktop/actor/pt_retrieval.py:114: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()



Already ran bm25_monot5 -> fln_praksis_2026_queries_topic_names_qid_query_sample30_bm25_monot5.csv (skipping)
Already ran splade -> fln_praksis_2026_queries_topic_names_qid_query_sample30_splade.csv (skipping)
Already ran e5 -> fln_praksis_2026_queries_topic_names_qid_query_sample30_e5.csv (skipping)
Already ran qwen3 -> fln_praksis_2026_queries_topic_names_qid_query_sample30_qwen3.csv (skipping)


{'bm25': 'fln_praksis_2026_queries_topic_names_qid_query_sample30_bm25.csv',
 'bm25_monot5': 'fln_praksis_2026_queries_topic_names_qid_query_sample30_bm25_monot5.csv',
 'splade': 'fln_praksis_2026_queries_topic_names_qid_query_sample30_splade.csv',
 'e5': 'fln_praksis_2026_queries_topic_names_qid_query_sample30_e5.csv',
 'qwen3': 'fln_praksis_2026_queries_topic_names_qid_query_sample30_qwen3.csv'}

## 9. Pool results into one deduplicated candidate set

`pool_results.py`'s `ResultPooler`: unions every method's results, keyed by
(qid, docno), for judging. Reusing the existing pool/labels here, since
those are already fully judged (see the note in step 10).

In [10]:
from pool_results import PoolingConfig, ResultPooler

if not step_done(POOL_CSV):
    config = PoolingConfig(dataset="fln", results_glob=f"{os.path.splitext(SAMPLE_TOPICS_CSV)[0]}_*.csv", output_csv=POOL_CSV)
    ResultPooler(config).run()
else:
    print(f"Already pooled -> {POOL_CSV} (skipping)")

df_pool = pd.read_csv(POOL_CSV, dtype={"qid": str, "docno": str})
print(f"{len(df_pool)} unique (qid, docno) pairs in the pool")
df_pool.head(3)

Already pooled -> fln_praksis_2026_queries_topic_names_qid_query_sample30_pool.csv (skipping)
5866 unique (qid, docno) pairs in the pool


,qid,query,docno,text,rank,score
0,103,health notifications and claimant status,bulg20263_chunk024,"Applicants who are children, unaccompanied children, disabled, elderly, preg...",0,-10.024446
1,103,health notifications and claimant status,dub-fran20262_chunk037,"In Lyon, Marseille, Paris and its surroundings, no subsequent claimants can ...",1,-13.255333
2,103,health notifications and claimant status,dub-belg20261_chunk023,Health care and a dignified standard of living should be always ensured. Acc...,2,-13.338537


## 10. LLM-as-judge relevance labelling

`judge_pool.py`'s `QrelsJudge`, via a local Ollama model (`llama3.1:8b`)
and the prompt in `qrels_prompt.txt` (0-5 relevance scale).

**Reusing the existing labels file**: judging this 30-topic pool took
~3h20m the first time (one LLM call per pooled row). The existing
`*_pool_labels.csv` already covers every row the *original* 3-method pool
produced (bm25_monot5, e5, splade); the BM25 run added in step 8 is new and
introduces a few (qid, docno) pairs the judge hasn't seen. Rather than
re-running the full 3-hour job, this evaluates every retriever (BM25
included) against the existing labels as-is -- a retriever's results don't
need to have been part of the original judged pool to be scored against it,
that's standard practice in IR evaluation (an unjudged retrieved document is
conventionally treated as not relevant).

In [11]:
from judge_pool import JudgeConfig, QrelsJudge

if not step_done(LABELS_CSV):
    print("Judging the pool with a local LLM (this can take hours on a large pool)...")
    config = JudgeConfig(input_csv=POOL_CSV, output_csv=LABELS_CSV)
    QrelsJudge(config).run()
else:
    print(f"Already judged -> {LABELS_CSV} (skipping; see note above)")

df_labels = pd.read_csv(LABELS_CSV, dtype={"qid": str, "docno": str})
print(f"{len(df_labels)} judged (qid, docno) pairs")
df_labels["label"].value_counts().sort_index()

Already judged -> fln_praksis_2026_queries_topic_names_qid_query_sample30_pool_labels.csv (skipping; see note above)
5862 judged (qid, docno) pairs


label
0     155
1     289
2    1193
3      86
4    3957
5     182
Name: count, dtype: int64

## 11. Evaluate every method: MAP@100, NDCG@10, MRR@10, P@5, R@5

`evaluate_run.py`'s `RunEvaluator`, via `pt.Evaluate`.

In [12]:
from evaluate_run import EvaluationConfig, RunEvaluator

metrics_rows = []
for retriever, results_path in result_paths.items():
    config = EvaluationConfig(
        dataset="fln", retriever=retriever, results_csv=results_path, labels_csv=LABELS_CSV,
        output_csv=f"{os.path.splitext(SAMPLE_TOPICS_CSV)[0]}_{retriever}_metrics.csv",
    )
    metrics_rows.append(RunEvaluator(config).run())

metrics_df = pd.concat(metrics_rows, ignore_index=True).sort_values("AP@100", ascending=False).reset_index(drop=True)
metrics_df.to_csv(f"{os.path.splitext(SAMPLE_TOPICS_CSV)[0]}_all_methods_metrics.csv", index=False)
metrics_df

,dataset,retriever,AP@100,nDCG@10,RR@10,P@5,R@5
0,fln,splade,0.533153,0.821088,1.000000,0.993333,0.027363
1,fln,e5,0.511622,0.802437,1.000000,0.973333,0.026877
2,fln,bm25_monot5,0.369531,0.775206,1.000000,0.980000,0.027113
3,fln,bm25,0.367685,0.735096,1.000000,0.973333,0.026933
4,fln,qwen3,0.130770,0.387878,0.474061,0.386667,0.010924
